# LLaMA-XR Extended -- Phase 0 + 1 (Colab)

Runs data prep (frozen DenseNet-121 -> 36-dim scores -> Alpaca prompts) and baseline
QLoRA fine-tuning of LLaMA 3.1 8B, matching the paper's setup.

**Before running:** Runtime -> Change runtime type -> GPU (A100 if you have Colab Pro,
T4 otherwise -- it'll just be slower).

## 1. Install dependencies

In [ ]:
!pip install -q torchxrayvision pandas
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft bitsandbytes accelerate nltk rouge-score

## 2. Clone your code repo

Push the `llama-xr-extended` project (the code only -- not the dataset) to your own
GitHub repo first, then clone it here. Replace the URL below with yours.

In [ ]:
!git clone https://github.com/<your-username>/llama-xr-extended.git
%cd llama-xr-extended

## 3. Mount Google Drive and get the dataset onto the Colab VM's local disk

Upload the R2Gen IU X-ray zip (the single `.zip` file, not the unzipped folder --
thousands of individual PNGs upload far slower than one archive) to your Google Drive
first, e.g. to `My Drive/llama-xr-extended/iu_xray.zip`. Then run this cell -- it mounts
your Drive and unzips straight into `data/iu_xray/` inside the cloned repo, so every path
in the scripts below just works with no edits.

Unzipping onto the VM's local disk (rather than reading PNGs directly off the mounted
Drive folder) matters here -- Drive-mounted I/O is slow for thousands of small files,
and training will otherwise bottleneck on file access rather than the GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this path to match where you put the zip in your Drive:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/llama-xr-extended/iu_xray.zip"

!mkdir -p data/iu_xray
!unzip -q "$DRIVE_ZIP_PATH" -d data/iu_xray
!ls data/iu_xray   # sanity check: should show images/ and annotation.json

## 4. Phase 0 -- build the Alpaca-format training data

In [ ]:
from src.data_prep import prepare_dataset

prepare_dataset(
    annotation_json="data/iu_xray/annotation.json",
    images_dir="data/iu_xray/images",
    output_dir="data",
)

## 5. Phase 1 -- QLoRA fine-tune LLaMA 3.1 8B

Uses `train_baseline_v2.py`, not the original `train_baseline.py`. The paper's exact
stated learning rate (2e-6) produced a model that never learned the task -- see
FINDINGS.md for the full diagnosis. `train_baseline_v2.py` fixes this (learning rate
raised to 2e-4) and is what actually produces a working model; `train_baseline.py` is
kept in the repo as a record of the original attempt, not something to run.

In [ ]:
!python src/train_baseline_v2.py

## 6. Generate reports on the held-out test set, then evaluate

Run inference with the saved adapter to produce `generated_reports.json` in the same
format as `data/sample_reports.json` (id, reference_report, generated_report,
classifier_scores), then run all three evaluations:

In [ ]:
!python src/generate_reports.py
!python src/eval_lexical.py generated_reports.json
!python src/eval_clinical.py generated_reports.json
!python src/hallucination_check.py generated_reports.json